# Stage 03: Vessel Segmentation

**Status:** Implemented -- pretrained LWNet, inference only. See `SEGMENTATION_ARCHITECTURE.md` Sec 1.2/2 for the full design and its Appendix A.1 for why this reverses an earlier, since-superseded "Baseline U-Net trained within this project" design.

## Objective

Run the pretrained [LWNet](https://github.com/agaldran/lwnet) ("The Little W-Net That Could", Galdrán et al., MIT License) vessel segmentation checkpoint over Stage 02's processed RGB images to produce a single-channel vessel probability map. Unlike every other trainable stage in this pipeline, **this stage has no training workflow of its own** -- it loads a checkpoint the LWNet authors already trained on DRIVE, outside this project entirely.

## Expected Inputs

- Stage 02 processed RGB images (`datasets/<name>/processed/`, Gamma Correction + CLAHE, native resolution -- see `PROJECT_CODE.md`'s Stage 02 Preprocessing Policy). This notebook does **not** run Stage 02 -- it only reads its already-generated output.
- The vendored LWNet checkpoint, uploaded to Google Drive at `exported_models/VesselSegmentation/best_model.pth` + `config.cfg` **before** this notebook is run for the first time (one-time manual step -- the checkpoint is not committed to git, mirroring how this repo excludes other trained-model binaries; see `.gitignore` and Section 4 below).

## Expected Outputs

- A `(H, W, 1)` vessel probability map per image, values in `[0, 1]`, matching that image's own `(H, W)` exactly -- consumed directly by Stage 04 (`SEGMENTATION_ARCHITECTURE.md` Sec 7.1's in-memory inference workflow; no per-dataset vessel-mask folder is created, matching the frozen architecture's design).
- Qualitative smoke-test visualizations (original / probability overlay / binary mask) for manual review this session.

## Datasets

None. Vessel Segmentation trains on nothing in this project -- DRIVE, CHASE_DB1, and STARE are explicitly **not** staged, downloaded, or referenced anywhere in this notebook (`SEGMENTATION_ARCHITECTURE.md` Sec 1.1). Section 6 below reads a small sample of already-processed images from whichever project datasets (EyeQ/APTOS2019/IDRiD) already have Stage 02 output on Drive, purely to smoke-test this stage -- it does not train or evaluate against them.

## Dependencies

Stage 02 (Image Preprocessing) must have already been run for at least one dataset -- this notebook reads its output, never regenerates it. First PyTorch-based stage in this repo (`torch`, `torchvision`, `scikit-image`, `scipy` -- added to `requirements.txt`); every other stage is TensorFlow/Keras.

## Workflow

Bootstrap -> Setup -> Environment Verification -> Checkpoint Staging -> Checkpoint Verification -> Dataset/Input Discovery -> Inference Smoke Test -> Output Verification -> Export -> Summary. Deliberately **not** Stage 01/02's Setup -> Verification -> Dataset Staging -> Training -> Evaluation -> Export shape -- this stage has no dataset to stage and no training loop (`colab/README.md`'s notebook table).

### 1. Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

### 2. Setup

Mounts Drive, installs `requirements.txt` (now including `torch`/`torchvision`/`scikit-image`/`scipy` for this stage), `cd`s into the repository. Identical call to every other stage notebook -- `setup.py` is not stage-specific.

In [ ]:
import setup

setup_info = setup.setup()
print(setup_info)

### 3. Environment Verification

`require_gpu=False`, unlike Stage 01's training notebook -- this stage's model is tiny (~68k parameters) and runs a handful of forward passes per image; a GPU speeds up large-batch runs but is not required to use this stage correctly. Everything else (Python/TensorFlow version, Drive mount, required packages including the new PyTorch stack) is still checked and still aborts on failure.

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=False,
)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### 4. Checkpoint Staging

Copies **only** the two vendored LWNet artifacts (`best_model.pth`, `config.cfg`) from their stable Drive location to this project's own `models/vessel_segmentation/` -- not the whole `external/lwnet/` repository, and not a full dataset staged through `dataset_staging.py` (that module stages *datasets*; a single ~900KB model file needs no thread-pool/verification machinery built for tens of thousands of files).

**One-time manual prerequisite:** upload `best_model.pth` (originally `external/lwnet/experiments/wnet_drive/model_checkpoint.pth`) and `config.cfg` to `MyDrive/DiabeticRetinopathy/exported_models/VesselSegmentation/` before running this cell for the first time -- the checkpoint is not committed to git (`.gitignore`), mirroring how this repo already excludes other trained-model binaries.

In [ ]:
import shutil

import config

VESSEL_SEG_DRIVE_DIR = colab_config.DRIVE.exported_model_dir("VesselSegmentation")
CHECKPOINT_FILES = ("best_model.pth", "config.cfg")

os.makedirs(config.VESSEL_SEG_MODEL_DIR, exist_ok=True)
staged_paths = {}
for filename in CHECKPOINT_FILES:
    src = os.path.join(VESSEL_SEG_DRIVE_DIR, filename)
    dst = os.path.join(config.VESSEL_SEG_MODEL_DIR, filename)
    if not os.path.isfile(src):
        raise RuntimeError(
            f"Vessel Segmentation artifact not found on Drive: {src}. Upload the vendored "
            "LWNet checkpoint there first -- see this cell's markdown and "
            "SEGMENTATION_ARCHITECTURE.md Sec 6."
        )
    shutil.copy2(src, dst)
    staged_paths[filename] = dst
    print(f"Staged {src} -> {dst} ({os.path.getsize(dst):,} bytes)")

### 5. Checkpoint Verification

Loads the staged checkpoint through this project's own `vessel_segmentation_model.py` / `vessel_segmentation_inference.py` (not ad hoc code) and verifies: parameter count matches LWNet's own published ~70k, `model.mode == "eval"` (a plain instance attribute, unrelated to `nn.Module.eval()` -- see both modules' docstrings for why this specific bug is easy to reintroduce), and that a dummy forward pass returns a single prediction tensor rather than the `(x1, x2)` tuple `WNet.forward()` returns when `mode != "eval"`.

In [ ]:
import torch

from vessel_segmentation_inference import DEFAULT_MODEL_PATH, load_vessel_model

vessel_model = load_vessel_model(DEFAULT_MODEL_PATH)
n_params = sum(p.numel() for p in vessel_model.parameters())
print(f"Loaded checkpoint from {DEFAULT_MODEL_PATH}")
print(f"Parameter count: {n_params:,} (LWNet's own paper reports ~70k for this configuration)")
print(f"model.mode = {vessel_model.mode!r} (must be 'eval')")
print(f"model.training (nn.Module mode) = {vessel_model.training} (must be False)")
assert vessel_model.mode == "eval", "model.mode was not set to 'eval' -- see vessel_segmentation_model.py"
assert vessel_model.training is False, "model.eval() was not called -- see vessel_segmentation_model.py"

dummy = torch.rand(1, 3, 512, 512)
with torch.no_grad():
    dummy_output = vessel_model(dummy)
assert isinstance(dummy_output, torch.Tensor), (
    f"Expected a single prediction tensor, got {type(dummy_output)} -- "
    "model.mode was not 'eval', so forward() returned the (x1, x2) training tuple."
)
assert dummy_output.shape == (1, 1, 512, 512)
print("Verified: forward() returns a single (1, 1, 512, 512) tensor, not a (x1, x2) tuple.")

### 6. Dataset / Input Discovery

Reads a small sample of already-processed (Stage 02 output) images directly from Drive -- **does not** run Stage 02 again, stage any dataset locally, or touch DRIVE/CHASE_DB1/STARE (not project datasets; `SEGMENTATION_ARCHITECTURE.md` Sec 1.1). Discovery is dataset-agnostic, mirroring `stage02_preprocessing.ipynb`'s own `discover_preprocessing_targets()` approach: walk for any `processed/` folder containing images, rather than hardcoding dataset names -- so this works whether Stage 02 has been run for EyeQ, APTOS2019, IDRiD (any/all of its three subtasks), or a future dataset, with no notebook changes.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
MAX_IMAGES_PER_DATASET = 2

def discover_processed_images(datasets_root, max_per_dataset=MAX_IMAGES_PER_DATASET):
    found = {}
    for dirpath, dirnames, filenames in os.walk(datasets_root):
        if os.path.basename(dirpath) != "processed":
            continue
        images = sorted(f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS))
        if not images:
            continue
        label = os.path.relpath(dirpath, datasets_root)
        found[label] = [os.path.join(dirpath, name) for name in images[:max_per_dataset]]
    return found

processed_by_dataset = discover_processed_images(colab_config.DRIVE.datasets_root)

if not processed_by_dataset:
    raise RuntimeError(
        f"No Stage 02 'processed/' output found under {colab_config.DRIVE.datasets_root}. "
        "Run colab/notebooks/stage02_preprocessing.ipynb for at least one dataset first -- "
        "this stage reads Stage 02's output, it does not generate it."
    )

for label, paths in processed_by_dataset.items():
    print(f"[{label}] {len(paths)} sample image(s): {[os.path.basename(p) for p in paths]}")

sample_images = [path for paths in processed_by_dataset.values() for path in paths]
print(f"\nTotal sample images discovered across {len(processed_by_dataset)} dataset(s): {len(sample_images)}")

### 7. Inference Smoke Test

Runs Stage 03 on exactly one discovered image and checks every property required before trusting this stage's output: the model loads, no shape/device errors, the probability map is finite, within `[0, 1]`, matches the input image's own `(H, W)` exactly, and is neither entirely zero nor entirely one.

In [ ]:
import time

from vessel_segmentation_inference import predict_vessel_mask
import numpy as np

smoke_test_image = sample_images[0]
print("Smoke-testing:", smoke_test_image)

t0 = time.perf_counter()
smoke_result = predict_vessel_mask(smoke_test_image, model=vessel_model)
wall_seconds = time.perf_counter() - t0

prob = smoke_result["probability_map"]
binm = smoke_result["binary_mask"]

checks = {
    "finite": bool(np.isfinite(prob).all()),
    "in_range_0_1": bool(prob.min() >= 0.0 and prob.max() <= 1.0),
    "shape_matches_input": prob.shape[:2] == smoke_result["input_shape"] == binm.shape[:2],
    "not_all_zero": not bool(np.all(prob == 0)),
    "not_all_one": not bool(np.all(prob == 1)),
}
for name, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")
assert all(checks.values()), "Stage 03 smoke test failed -- see failing check(s) above."

print(f"\ninput_shape={smoke_result['input_shape']}  probability_map.shape={prob.shape}")
print(f"prob min/max/mean = {prob.min():.4f} / {prob.max():.4f} / {prob.mean():.4f}")
print(f"tta_used={smoke_result['tta_used']}  threshold_used={smoke_result['threshold_used']} ({smoke_result['threshold_source']})")
print(f"inference_seconds={smoke_result['inference_seconds']:.2f}  wall_seconds={wall_seconds:.2f}")
print("Smoke test PASSED.")

### 8. Output Verification

Qualitative check (a visualization -- original / vessel-probability overlay / binary mask), then a small cross-dataset quantitative pass over every sample image discovered in Section 6, recording shape, timing, and probability statistics per image. Purely for manual review this session -- see Section 9 for why nothing here is written back to Drive.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

def visualize_prediction(image_path, result, title):
    original = np.array(Image.open(image_path).convert("RGB"))
    prob2d = result["probability_map"][..., 0]
    alpha = prob2d[..., None]
    red = np.zeros_like(original, dtype=np.float32)
    red[..., 0] = 255
    overlay = (original.astype(np.float32) * (1 - 0.6 * alpha) + red * (0.6 * alpha)).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original); axes[0].set_title("Stage 02 processed"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("Vessel probability overlay"); axes[1].axis("off")
    axes[2].imshow(result["binary_mask"][..., 0], cmap="gray")
    axes[2].set_title(f"Binary mask (t={result['threshold_used']})"); axes[2].axis("off")
    fig.suptitle(title)
    plt.show()
    return fig

visualize_prediction(smoke_test_image, smoke_result, f"Stage 03 smoke test -- {os.path.basename(smoke_test_image)}")

cross_dataset_records = []
for label, paths in processed_by_dataset.items():
    for path in paths:
        result = predict_vessel_mask(path, model=vessel_model)
        cross_dataset_records.append({
            "dataset": label,
            "image": os.path.basename(path),
            "input_shape": result["input_shape"],
            "prob_min": float(result["probability_map"].min()),
            "prob_max": float(result["probability_map"].max()),
            "prob_mean": float(result["probability_map"].mean()),
            "foreground_pct": float(100.0 * result["binary_mask"].sum() / result["binary_mask"].size),
            "inference_seconds": result["inference_seconds"],
        })

cross_dataset_df = pd.DataFrame(cross_dataset_records)
print(cross_dataset_df.to_string(index=False))

### 9. Export

There is nothing for this stage to export back to Drive: it consumes a checkpoint (Section 4), it does not produce one, and Stage 03's output (a vessel probability map) is designed to be consumed **in memory** by Stage 04, not persisted per-dataset (`SEGMENTATION_ARCHITECTURE.md` Sec 7.1 -- `vessel_stage.predict(preprocessed_image)` feeds directly into `lesion_stage.predict(...)`). This cell only re-confirms the staged checkpoint this session actually used is the one now sitting at `config.VESSEL_SEG_MODEL_DIR`, so Stage 04's own notebook can rely on it being in place.

In [ ]:
print("Stage 03 checkpoint ready for downstream use at:")
for filename, path in staged_paths.items():
    print(f"  {filename}: {path} ({os.path.getsize(path):,} bytes)")
print("\nNo Drive export needed -- this stage produces no training output. "
      "Stage 04's notebook should call vessel_segmentation_inference.load_vessel_model() "
      "the same way this notebook did.")

### 10. Summary

In [ ]:
print("=" * 70)
print("Stage 03: Vessel Segmentation -- Summary")
print("=" * 70)
print(f"Model: pretrained LWNet (wnet, in_c=3, n_classes=1, layers=(8,16,32))")
print(f"Checkpoint: {DEFAULT_MODEL_PATH} ({n_params:,} parameters)")
print(f"Trained by: LWNet's original authors on DRIVE -- NOT trained within this project")
print(f"TTA: {config.VESSEL_SEG_TTA} (config.VESSEL_SEG_TTA / VESSEL_SEG_TTA env var)")
print(f"Binarizing threshold: {smoke_result['threshold_used']} -- LWNet's own DRIVE-tuned reference "
      "default, NOT validated against this project's datasets (SEGMENTATION_ARCHITECTURE.md Sec 2.3)")
print(f"Datasets smoke-tested: {list(processed_by_dataset.keys())}")
print(f"Images tested: {len(cross_dataset_records)}")
print(f"Mean inference time per image: {cross_dataset_df['inference_seconds'].mean():.2f}s "
      "(CPU; a GPU runtime will be substantially faster for large-batch runs)")
print()
print("Known caveat: LWNet never sees Gamma/CLAHE in its own training pipeline (only resize + "
      "ToTensor, no mean/std normalization) -- Stage 02's contrast-enhanced output is a documented "
      "distribution shift relative to what this checkpoint was tuned against. Visually inspect "
      "Section 8's overlays before trusting this stage's output for any downstream decision.")
print()
print("external/lwnet/ is no longer required for this stage to run -- vessel_segmentation_model.py "
      "and vessel_segmentation_inference.py are self-contained, vendoring only what Sections 2-3 "
      "of the integration inspection identified as required.")